In [2]:
%matplotlib inline

In [3]:
# 2
# ── INSTALACIÓN DE DEPENDENCIAS (ejecutar una sola vez al inicio de la sesión) ──
# Si Colab pide "Restart runtime", hacerlo antes de continuar con las siguientes celdas.
!pip install -q tifffile natsort shapely
# Fix compatibilidad PyTorch + sympy
!pip install -q sympy==1.13.1

import importlib
required = ['tifffile', 'natsort', 'shapely', 'cv2', 'sklearn', 'tqdm']
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  FALTA {pkg}")
print("Dependencias verificadas.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 55.0 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.
  OK tifffile
  OK natsort
  OK shapely
  OK cv2
  OK sklearn
  OK tqdm
Dependencias verificadas.


## Aprendizaje Profundo para el Análisis de Imágenes
## Grados en Ingeniería de Telecomunicación

# PROYECTO 2A:
# SEGMENTACIÓN DE IMÁGENES CON DEEPLAB-V3

<center><img src='https://www.uc3m.es/ss/Satellite?blobcol=urldata&blobkey=id&blobtable=MungoBlobs&blobwhere=1371614316787&ssbinary=true' width=400 /></center>

El propósito de este proyecto es que el estudiantado se familiarice con una red neuronal convolucional (CNN) para segmentación semántica de imágenes (en la que a cada píxel de la imagen de entrada se asocia una clase de objeto). En concreto, en este tutorial se realizará _fine-tuning_ sobre la red `Deeplab-V3`. El objetivo del proyecto es que el estudiantado analice los bloques específicos para segmentación de la arquitectura de red y trabaje sobre otros conceptos como el _data augmentation_ o la función de coste de la red neuronal en un escenario real con el objetivo de mejorar el aprendizaje de la misma.

En concreto, se realizará _fine-tuning_ sobre la red `Deeplab-V3`, una de las redes del estado del arte para segmentación semántica. En nuestro caso, la primera tarea de segmentación a la que se va a aplicar esta arquitectura es la segmentación de objetos en imágenes convencionales, utilizando un subconjunto de la base de datos PascalVOC2012.

Este tutorial es una adaptación del tutorial disponible en https://expoundai.wordpress.com/2019/08/30/transfer-learning-for-segmentation-using-deeplabv3-in-pytorch/.

Referencias:
- [1] Deeplab-V3. https://arxiv.org/abs/1706.05587
- [2] Imagen. https://raw.githubusercontent.com/abreheret/PixelAnnotationTool/master/images_test/Abbey_Road.jpg
- [3] Imagen. https://raw.githubusercontent.com/abreheret/PixelAnnotationTool/master/images_test/Abbey_Road_color_mask.png
- [4] Deeplab. https://arxiv.org/abs/1606.00915
- [5] PASCAL VOC 2012 dataset. http://host.robots.ox.ac.uk/pascal/VOC/voc2012/index.html
- [6] Regularización local. http://www.sfu.ca/~abentaie/papers/miccai16.pdf
- [7] Focal Loss for Object Detection. https://arxiv.org/abs/1708.
- [8] Dice Loss. https://arxiv.org/abs/1606.04797

## Antes de empezar

Antes de empezar, en caso de que vaya a utilizar Google Colab, necesita descomprimir los archivos de la práctica en una carpeta en Drive y cambiar el directorio de trabajo al de dicha carpeta. Para ello, ejecute el siguiente código:

In [4]:
# ────────────────────────────────────────────────────────────────────────────
# SETUP KAGGLE: Clonar el repo del proyecto (Sin Google Drive)
# ────────────────────────────────────────────────────────────────────────────
import os, sys, subprocess

# 1) Configuración de GitHub
REPO_URL  = "https://github.com/100495891/apai-pr2a-xbd.git"
REPO_PATH = "/kaggle/working/repo"
BRANCH    = "feat/losses"   # <--- ¡Ya te lo dejo puesto en tu rama!

# 2) Clonar el repo del proyecto desde GitHub (o pull si ya está)
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

subprocess.run(["git", "-C", REPO_PATH, "checkout", BRANCH], check=True)

# 3) Añadir el repo al sys.path para poder importar los módulos .py
sys.path.insert(0, REPO_PATH)
print(f"Repo en {REPO_PATH}, rama actual: {BRANCH}")

Already up to date.
Branch 'feat/losses' set up to track remote branch 'feat/losses' from 'origin'.
Repo en /kaggle/working/repo, rama actual: feat/losses


Switched to a new branch 'feat/losses'


In [5]:
# EJECUTAR SOLO UNA VEZ Y VERIFICAR CON LA SIGUIENTE CELDA

import os

print("\n── Contenido de la carpeta de Datos de Kaggle ──")
!ls -la /kaggle/input/

print("\n── Buscando las carpetas del dataset ──")
# Esto buscará todas las carpetas dentro de input para que veamos 
# cómo se llaman exactamente tus datasets de xBD
!find /kaggle/input/ -maxdepth 3 -type d | sort | head -30
"""
# ── Descomprimir datasets ──────────────────────────────────────────────────
# (solo si aún no están descomprimidos; el flag -n evita sobreescribir)

print("Descomprimiendo xBD_UC3M ...")
!unzip -q -n "xBD_UC3M.zip"

# ── Verificar estructura esperada ──────────────────────────────────────────
print("\n── Contenido raíz ──")
!ls -la

print("\n── Estructura xBD_UC3M ──")
!find xBD_UC3M -maxdepth 3 -type d | sort | head -30
"""


── Contenido de la carpeta de Datos de Kaggle ──
total 12
drwxr-xr-x 3 root root 4096 Apr 29 07:31 .
drwxr-xr-x 5 root root 4096 Apr 29 07:14 ..
drwxr-xr-x 3 root root 4096 Apr 29 07:31 datasets

── Buscando las carpetas del dataset ──
/kaggle/input/
/kaggle/input/datasets
/kaggle/input/datasets/mariamontanet
/kaggle/input/datasets/mariamontanet/xbd-uc3m


'\n# ── Descomprimir datasets ──────────────────────────────────────────────────\n# (solo si aún no están descomprimidos; el flag -n evita sobreescribir)\n\nprint("Descomprimiendo xBD_UC3M ...")\n!unzip -q -n "xBD_UC3M.zip"\nprint("Descomprimiendo xBD_UC3M_test_with_labels ...")\n!unzip -q -n "xBD_UC3M_test_with_labels.zip"\n\n# ── Verificar estructura esperada ──────────────────────────────────────────\nprint("\n── Contenido raíz ──")\n!ls -la\n\nprint("\n── Estructura xBD_UC3M ──")\n!find xBD_UC3M -maxdepth 3 -type d | sort | head -30\n\nprint("\n── Estructura xBD_UC3M_test_with_labels ──")\n!find xBD_UC3M_test_with_labels -maxdepth 3 -type d 2>/dev/null | sort | head -20\n# (si falla, prueba con el nombre exacto que aparezca en \'ls -la\')\n'

In [10]:
# ────────────────────────────────────────────────────────────────────────────
# SETUP KAGGLE: Comprobación rápida: contar archivos por split
# ────────────────────────────────────────────────────────────────────────────
import os

# Usamos la ruta exacta que nos dio Kaggle en la celda anterior
BASE_DIR = "/kaggle/input/datasets/mariamontanet/xbd-uc3m/xBD_UC3M"

print("── Comprobación rápida: contar archivos por split ──")
for split in ['train', 'val', 'test']:
    
    # Montamos la ruta completa. 
    # NOTA TECH LEAD: Si al ejecutar te dice "NO EXISTE", es porque el zip 
    # ha creado una subcarpeta. En ese caso cambia la línea de abajo por: 
    # path = f"{BASE_DIR}/xBD_UC3M/{split}"
    path = f"{BASE_DIR}/{split}"
    
    if os.path.exists(path):
        n_tif = !find {path} -name "*.tif" | wc -l
        n_json = !find {path} -name "*.json" | wc -l
        print(f"  {split}: {n_tif[0].strip()} imágenes .tif, {n_json[0].strip()} labels .json")
    else:
        print(f"  {split}: NO EXISTE en {path} — revisar estructura")

── Comprobación rápida: contar archivos por split ──
  train: 512 imágenes .tif, 512 labels .json
  val: 90 imágenes .tif, 90 labels .json
  test: 126 imágenes .tif, 126 labels .json


Además, si quiere ejecutar el código con soporte a GPU, en Google Colab vaya a `Entorno de ejecución->Cambiar tipo entorno de ejecución` y seleccione GPU en `acelerador por hardware`.

## Parte 1. Fundamento teórico

### Segmentación de objetos

Frente a los métodos tradicionales de segmentación de imágenes (*thresholding, clustering* o *region growing*), las técnicas de *deep learning* han demostrado ser mucho más eficaces para tareas complejas de segmentación de imágenes. Sin embargo, estas técnicas requieren grandes bases de datos anotadas píxel a píxel para su entrenamiento, lo que supone un gran esfuerzo de anotación.

El objetivo de un algoritmo de segmentación de objetos es generar máscaras de salida a nivel de píxel en las cuales a las regiones que pertenecen a ciertas categorías se les asigna el mismo valor de píxel. Si se codifican en color (asignando un color diferente a cada clase de objetos) se obtienen resultados como los que se muestran en la siguiente figura [2-3], donde en azul se representa la clase vehículo, en rojo la clase persona, etc.

<table><tr><td><img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/beatles.jpg"></td><td><img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/beatles_mask.png"></td></tr></table>

Por tanto, como entrada a nuestro algoritmo de segmentación se tendrá un conjunto de imágenes y sus correspondientes máscaras *ground truth* píxel a píxel.

### Contexto vs resolución

El principal reto en segmentación semántica es encontrar un compromiso entre la importancia del contexto global de la imagen (para segmentar un objeto, es necesario identificar sus distintas partes y diferenciarlo del resto de objetos de la imagen), y las características locales de la imagen (una segmentación precisa ha de analizar los valores de los píxeles que se encuentran en los alrededores del objeto para delimitar la frontera del mismo). La siguiente figura muestra las distintas estrategias que se utilizan para intentar representar este compromiso sobre CNNs.

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/arquitecturas.png">

La aproximación tradicional que se ha seguido para utilizar extraer características de alto nivel de la imagen ha sido el análisis de la misma a través de una representación en pirámide, en la que se consideran la imagen original y versiones de menor tamaño de la misma (filtrado gaussiano y reducción del tamaño mediante submuestreo). Si se aplica el mismo procesado a todos los niveles de la pirámide, se obtienen características de bajo nivel en los primeros niveles, y de alto nivel conforme se va profundizando en la misma (véase la figura a) arriba, donde se puede ver que se mezclan las características de dos pirámides de imagen que parten de distintas escalas). De esta manera, en las versiones de menor tamaño pierde importancia la información local de la imagen en favor de la información global, y viceversa. Las arquitecturas de redes neuronales convolucionales (FCNs) de clasificación se basan en esta representación para intentar resumir la información de la imagen (las características a niveles menos profundos sirven para calcular características a niveles más profundos) y proporcionar su categoría.

Por el contrario, en segmentación no solo importa el contexto global de la imagen, sino que las características locales también cuentan. Las redes totalmente convolucionales para segmentación (*Fully Convolutional Networks*, FCNs) introducen una estructura *encoder-decoder* en la red para obtener una salida con una resolución igual (o lo más cercana posible) a la de la entrada (véase la figura b) arriba). Para ello, el *encoder* reduce paulatinamente la dimensión de los mapas de características de manera que la información global se captura en las capas profundas; y el *decoder* parte de esta información global y paulatinamente recupera la dimensión original de la imagen. Frecuentemente se incluyen conexiones entre las capas de igual dimensión del *encoder* y el *decoder* para facilitar el empleo directo de las características más locales en la salida final de la segmentación.

En las redes Deeplab [1]-[4] se hace uso de las dos estrategias de la derecha: convoluciones *atrous* y *Spatial Pyramid Pooling*. Estas estrategias se explican en la siguiente sección.


### Deeplab-V3

Aunque se recomienda echar un vistazo al artículo sobre Deeplab-V3 [1], en esta sección se van a explicar los conceptos más importantes de la red que son necesarios para el desarrollo de la práctica.

Deeplab-V3 es un _framework_ que adapta cualquier red convolucional dedicada a la clasificación (dada una imagen, encontrar una categoría que describa el contenido total de la imagen) a la segmentación de objetos. Para ello, a partir de un _backbone_ inicial (las capas destinadas a la extracción de características) de cualquier red de clasificación, propone una serie de capas y bloques destinados a extraer características de contexto en la imagen sin comprometer la resolución de los mapas de características (sin reducir aun más su resolución). Para ello, hace uso de las dos estrategias que se describen a continuación.

#### Convoluciones *atrous*

La contribución principal de la familia de redes Deeplab [4] es el diseño de las convoluciones *atrous* o convoluciones *dilated*. Este tipo de convoluciones se utilizan para reemplazar la estrategia tradicional de reducir el tamaño de la imagen a través de capas de *max pooling* con *stride* a lo largo de la red para obtener representaciones más globales del contenido de la imagen. Esta estrategia estándar hace que el *stride* acumulado a lo largo de la red sea muy elevado (32, por ejemplo, lo que significa que el tamaño de las características de una imagen original de $HxW$ es $H/32xW/32$), lo que puede ser contraproducente en segmentación (a pesar de obtener mejores características de contexto, la reducción de las dimensiones de la imagen hace que las segmentaciones sean menos precisas). Las convoluciones *atrous*, por su parte, mantienen la resolución de la entrada a la vez que extraen características de mayor orden a través del uso del *stride*. Es decir, para computar el valor de un cierto píxel $y[i]$ se toman los valores de los píxeles alejados del mismo $r$ posiciones en la entrada $x$ multiplicados por el elemento correspondiente del filtro $w$.

\begin{equation}y[i]=\sum_{k}{x[i+r\cdot k] w[k]}.\end{equation}

En modo filtro, una convolución *atrous* con tasa $r$ se consigue añadiendo $r-1$ ceros entre los elementos del filtro original, como se muestra en la siguiente figura [1]:

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/atrous.png" width="600pix">

A continuación se puede observar una animación en la que se comparan una convolución 2D tradicional con una convolución *atrous*, ambas con el mismo *field of view*, pero la primera usa 15 parámetros y la segunda 9 parámetros. Mientras que la convolución estándar usa un filtro de $5x5$, *stride* de 1, *dilate* de 1 y *padding* de 1; la convolución *atrous* usa un filtro de $3x3$, *stride* $r=2$, *dilate* de 2 y no usa *padding*.

<table><tr><td><img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/2dconv.gif"></td><td><img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/atrousconv.gif"></td></tr></table>

En la práctica, estas convoluciones no se utilizan a lo largo de toda la red: en primer lugar, porque reducir el tamaño de la imagen en las primeras capas de la red es útil desde el punto de vista del coste computacional (sería muy costoso trabajar con las imágenes a tamaño completo durante todo el procesado); y en segundo lugar, porque de hecho esta sustitución del *max pooling* por convoluciones *atrous* es más útil en capas profundas de la red (con menos tasa $r$ se recorren grandes porciones de imagen). Un ejemplo de flujo de trabajo sobre ResNet se muestra a continuación [1]:

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/atrous2.png">

Se puede ver como a partir del bloque 3 se utilizan convoluciones *atrous* __en cascada__ con tasa variable $2^n$ para sustituir al *max pooling* con tasa 2. De este modo se mantiene la resolución en valores aceptables.

#### *Atrous Spatial Pyramid Pooling*

La segunda contribución del artículo [1] tiene que ver con el uso de las convoluciones *atrous* como extractores de características __en paralelo__ con distinta tasa $r$, de manera que las características que se extraen en cada rama sean más globales o locales en función de dicha tasa. Esto es lo que se llama ASPP (*Atrous Spatial Pyramid Pooling*). Las características de cada rama se pueden agrupar bien mediante suma o bien mediante concatenación, para obtener después la salida final de segmentación de la red.

En concreto, la estructura del *frawework* Deeplab-V3 es la que se muestra a continuación:

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/aspp.png">

Se puede observar que se parte de un mapa de características de la imagen con *stride* 16 al que se añade una capa con convoluciones *atrous* con tasa $r=2$ y posteriormente el bloque ASPP. Este consta de los siguientes módulos:

- Una capa con una convolución 1x1 que extrae características más locales del mapa de entrada.
- Sendas convoluciones *atrous* con tasas $r=6$, $r=12$ y $r=18$, respectivamente, que extraen características con diferente contexto global de los mapas de entrada.
- Un *avg pooling* a nivel de mapa de entrada que genera como salida la media de cada canal. Esta característica global de la imagen permite ponderar la importancia de los distintos canales.

### Medidas de evaluación

Para evaluar la calidad en la segmentación de objetos, se suele utilizar la medida *Intersection over Union*, IoU, también denominada *Jaccard Index* (JI). La medida $IoU$ mide la similitud entre dos regiones $A$ y $B$ como:

\begin{equation}
IoU=\frac{A \cap B}{A \cup B}
\end{equation}

siendo $\cap$ la intersección entre las regiones (area común) y $\cup$ la unión o área total que cubren entre ambas. Se considera un umbral mínimo de IoU en torno a $IoU_{th}=0.7$ para considerar una detección como correcta. A continuación se puede ver un ejemplo de la medida $IoU$.

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/iou.png">


### Base de datos

La base de datos sobre la que se trabajará en la práctica es un subconjunto de la base de datos PASCAL VOC 2012 [5]. Esta base de datos contiene imágenes con diversos objetos pertenecientes a 20 categorías:
- __Person__: _person_
- __Animal__: _bird, cat, cow, dog, horse, sheep_
- __Vehicle__: _aeroplane, bicycle, boat, bus, car, motorbike, train_
- __Indoor__: _bottle, chair, dining table, potted plant, sofa, tv/monitor_

Las imágenes se proporcionan en sendas carpetas para entrenamiento y test. Los conjuntos se distribuyen de la siguiente manera:
- Entrenamiento: 234 imágenes.
- Test: 68 imágenes.

## Parte 2. Implementación

En primer lugar, se importan las librerías necesarias y se definen algunos parámetros generales.

In [ ]:
import os
import glob
from tqdm import tqdm
import numpy as np
import copy
import time
import torch
import torch.nn.functional as FT
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from torchvision.models.segmentation import deeplabv3_resnet101
from torchvision import transforms, utils
import torchvision.transforms.functional as F
from PIL import Image, ImageFile, ImageDraw
import cv2
import csv
import random
import matplotlib.pyplot as plt
# Set random seed for reproducibility
manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
# torch uses some non-deterministic algorithms
torch.backends.cudnn.enabled = False

### Entradas

Se definen algunas entradas para la ejecución:

-  **data_dir** - el directorio raíz de la base de datos, que se describe posteriormente.
-  **num_workers** - el número de hebras para cargar los datos con la clase DataLoader.
-  **img_size** - el tamaño de las imágenes de entrada a la red (cuadradas).
-  **batchsize_train** - el tamaño de _batch_ que se utiliza para entrenamiento.
-  **batchsize_test** - el tamaño de _batch_ que se utiliza para test.
-  **num_classes** - el número de clases que detectar.
-  **class_names** - los nombres de las clases que detectar.
-  **num_epochs** - número de _epochs_ para el entrenamiento de la red.
-  **step_size** - número de *epochs* tras los cuales se reduce el *learning rate* en un factor 0.1.
-  **lr** - *learning rate* inicial.
-  **result_dir** - el directorio raíz para almacenar los resultados.
-  **device** - el dispositivo (GPU o CPU) para la ejecución.

In [ ]:
data_dir = "myVOC4"
num_workers=8         # to debug, fix num_workers=0
img_size=512           # Size of the images
batchsize_train=1      # Batch size for training
batchsize_test=1       # Batch size for test (it must be one to generate predictions)
num_classes=21         # Number of classes
class_names = [
    'background',    # 0
    'aeroplane',     # 1
    'bicycle',       # 2
    'bird',          # 3
    'boat',          # 4
    'bottle',        # 5
    'bus',           # 6
    'car',           # 7
    'cat',           # 8
    'chair',         # 9
    'cow',           # 10
    'diningtable',   # 11
    'dog',           # 12
    'horse',         # 13
    'motorbike',     # 14
    'person',        # 15
    'pottedplant',   # 16
    'sheep',         # 17
    'sofa',          # 18
    'train',         # 19
    'tvmonitor'      # 20
]
num_epochs =8
step_size=5
result_dir = 'results' # Result directory
# Detect if we have a GPU available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### Red

La red a utilizar será una versión de Deeplab-V3 con ResNet-101 como *backbone*, ya pre-entrenada sobre el conjunto de entrenamiento de la base de datos COCO. En concreto, se utilizan los 4 primeros bloques de esta red, compuestos de capas convolucionales, normalización de batch (cuyos parámetros se congelan) y capas no lineales. Vea como los bloques se modifican añadiendo convoluciones *atrous* en lugar de *max pooling*, como se ha descrito en el apartado teórico, y a partir del cuarto bloque se añade el bloque de ASPP y la convolución que genera la salida de segmentación.

A continuación se incluye la función que construye la red. Analice la arquitectura.

- ¿Se realiza el submuestreo con capas de *max pooling* en las capas iniciales, y *layers* 1, 2 y 3? ¿Por qué?
- ¿Coinciden las tasas $r$ de las convoluciones *atrous* de los bloques 4 y el ASPP con el apartado teórico? ¿Por qué? ¿Cuál es el stride acumulado hasta la capa 4?
- ¿Cuál es la diferencia en cuanto a los parámetros entre una convolución *atrous* (mantiene el tamaño de la imagen) y las convolución que sustituyen al *max pooling*?

__**IMPORTANTE:__ no preste atención al `aux_classifier` de momento. Se trata de un clasificador auxiliar que se utiliza para mejorar el problema del *vanishing gradient*. Si se introduce una función de pérdida a su salida y se realiza la retropropagación, este bloque introduce gradientes más robustos en un punto intermedio de la red que pueden ayudar al entrenamiento (dado que la arquitectura es muy profunda, los gradientes de la salida estándar se van desvaneciendo a medida que se va avanzando desde la salida del clasificador estándar hasta el inicio de la red). Por el momento no se va a utilizar.

In [ ]:
def get_deeplabv3(num_classes=1):
    model = deeplabv3_resnet101(pretrained=True, progress=True)
    model.classifier = DeepLabHead(2048, num_classes)
    return model

In [ ]:
model = get_deeplabv3(num_classes)
print(model)
model.to(device)

__El modelo no es el mismo estudiado en el apartado teórico. En concreto hay una convolución con stride y max-pooling en el bloque 0 (rápido se reduce el tamaño en 4), en la layer 1 no existe downsample, en la layer 2 sí y en la layer 3 no, de manera que el stride acumulado hasta el bloque 4 es 8. Por ello, el bloque 4 contiene dos convoluciones *atrous*, una con $r=2$ y otra con $r=4$ (para aumentar el *receptive field* a la entrada del ASPP). Además, en el ASPP se consideran tasas $r=12, 24, 36$, el doble de las del apartado teórico__.

### Base de datos

A continuación se describe la base de datos. Para simplificar su tratamiento posterior, la base de datos se proporciona en forma de imágenes (`images`) y máscaras para cada una de las instancias de objetos (`instances`): es decir, para cada imagen se tienen tantas máscaras como objetos existan en la misma, con la segmentación de cada uno de ellos. Asimismo, la categoría de cada objeto va codificada en el nombre de la imagen. Además, se proporcionan la segmentación semántica `classes` y la segmentación de instancias `masks` por si resultan útiles.

Mediante este formato realizar el proceso de *data augmentation* es más sencillo, ya que basta con aplicar las mismas transformaciones (traslaciones, rotaciones, etc.) a la imagen original y a la máscara.

A continuación se define la clase que implementa la carga la base de datos.

In [ ]:
class myVOCDataset(object):
    def __init__(self, root, train, norm, data_augm):
        self.root = root
        self.train = train
        self.data_augm = data_augm
        # load all image files, sorting them to
        # ensure that they are aligned
        if (self.train):
            self.imgs = [os.path.join(root,name)
                 for root, dirs, files in os.walk(root)
                 for name in files if ('train'+os.path.sep+'images' in root)
                 if name.lower().endswith(".jpg")]
        else:
            self.imgs = [os.path.join(root,name)
                 for root, dirs, files in os.walk(root)
                 for name in files if ('test'+os.path.sep+'images' in root)
                 if name.lower().endswith(".jpg")]

        # Lookup Table (LUT)
        # PASCAL VOC uses 0-20 for classes and 255 for borders.
        self.lut = np.zeros(256, dtype=np.uint8)

        # Define your mapping: {Original Index: New Index}
        # 'background': 0 -> 0
        # 'bottle':     5 -> 1
        # 'chair':      9 -> 2
        # 'diningtable':11 -> 3
        # 'sofa':       18 -> 4
        new_mapping = {5: 1, 9: 2, 11: 3, 18: 4}

        for old_idx, new_idx in new_mapping.items():
            self.lut[old_idx] = new_idx

    def __getitem__(self, idx):
        ImageFile.LOAD_TRUNCATED_IMAGES = True

        # load images ad masks
        img_path = self.imgs[idx]
        img = Image.open(img_path).convert("RGB")

        mask_path = self.imgs[idx].replace('images','classes').replace('jpg','png')
        mask = Image.open(mask_path)
        mask = np.asarray(mask)
        #mask = self.lut[mask]

        # Resize all images to have the same shape and convert into tensor
        image = cv2.resize(np.asarray(img), (img_size,img_size), interpolation=cv2.INTER_AREA)
        image = F.to_tensor(image)
        mask = cv2.resize(mask.astype('long'), (img_size,img_size), interpolation=cv2.INTER_NEAREST)
        mask[mask==255]=0
        mask = F.to_tensor(mask).long().squeeze()

        sample = {'image': image, 'mask': mask, 'img_path': img_path}

        if (self.train and self.data_augm):
            # Implement your data augmentation
            pass

        return sample

    def __len__(self):
        return len(self.imgs)

Se carga la base de datos.

In [ ]:
image_mean=[0.485, 0.456, 0.406]
image_std=[0.229, 0.224, 0.225]

# use our dataset and defined transformations
dataset = myVOCDataset(data_dir, train=True, norm=transforms.Normalize(image_mean,image_std), data_augm=None)
dataset_test = myVOCDataset(data_dir, train=False, norm=transforms.Normalize(image_mean,image_std), data_augm=None)

# define training and test data loaders
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=batchsize_train, shuffle=True, num_workers=num_workers, # to debug, fix num_workers=0
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=batchsize_test, shuffle=False, num_workers=num_workers,
)

dataloaders = {'Train': data_loader, 'Val': data_loader_test, 'Test': data_loader_test}

### Medidas de evaluación

Asimismo, se define la función que testea la bondad de nuestro modelo de segmentación multiclase en términos de segmentación (índice Jaccard o *Intersection over Union*). El valor máximo de la función `softmax` a la salida de la red proporciona la clase predicha.

La función recibe como parámetros:

- __model__: la CNN que evaluar.
- __dataloader__: el cargador de los datos de test.
- __num_classes__: número de clases
- __class_names__: nombres de las clases de objetos a detectar (siempre hay que incluir en primer lugar la clase *background*).
- __result_dir__: el directorio donde guardar los resultados.
- __SAVE_OPT__: para guardar o no los resultados de test como imágenes con el *ground truth* en verde y la segmentación en rojo, en el directorio `predictions_binary`.
- __batchsize__: debe ser igual a 1.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.colors as mcolors

def test_segmentation_model(model, dataloaders, num_classes, class_names, result_dir, SAVE_OPT, batch_size=1):
    cm_total = np.zeros((len(class_names), len(class_names)), dtype=np.int64)  # Fix: cm must be 2D
    jaccard = []

    # Set up result directory and CSV file
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, 'results_multiclass.csv')
    if os.path.exists(csv_path):
        os.remove(csv_path)

    if SAVE_OPT:
        os.makedirs(os.path.join(result_dir, 'predictions_multiclass'), exist_ok=True)

    # --- CONFIGURACIÓN DE COLORES POR CLASE ---
    if num_classes <= 10:
        cmap_base = plt.get_cmap('tab10')
    elif num_classes <= 20:
        cmap_base = plt.get_cmap('tab20')
    else:
        cmap_base = plt.get_cmap('jet')

    colors_list = [cmap_base(i) for i in range(num_classes)]
    colors_list[0] = (0, 0, 0, 1.0) # Negro para el fondo
    custom_cmap = mcolors.ListedColormap(colors_list)
    norm_bins = np.arange(num_classes + 1) - 0.5
    custom_norm = mcolors.BoundaryNorm(norm_bins, num_classes)
    # ---------------------------------------------

    with open(csv_path, 'w', newline='') as csv_file:
        coord_writer = csv.writer(csv_file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)

        model.eval()
        for sample in tqdm(dataloaders['Test']):
            with torch.no_grad():
                inputs = sample['image'].to(device)
                masks = sample['mask'].to(device)

                y_prob = torch.nn.functional.softmax(model(inputs)['out'], dim=1)
                _, y_pred = torch.max(y_prob, dim=1)
                y_pred = y_pred.data.cpu().numpy()   # (B, H, W)
                y_true = masks.data.cpu().numpy()    # (B, H, W) — categorical, not one-hot

            for j in range(y_pred.shape[0]):
                img_name = os.path.basename(sample['img_path'][j])
                pred_j = y_pred[j]   # (H, W)
                true_j = y_true[j]   # (H, W)

                # Jaccard index per foreground class (excluding background at index 0)
                ji = np.zeros(num_classes - 1, dtype=float)
                for i in range(1, num_classes):
                    pred_i = pred_j == i
                    true_i = true_j == i
                    intersection = np.logical_and(pred_i, true_i).sum()
                    union = pred_i.sum() + true_i.sum() - intersection
                    ji[i - 1] = intersection / union if union > 0 else 0.0  # Fix: avoid division by zero
                jaccard.append(ji)

                # Confusion matrix — both arrays must be flat 1D
                cm_total += confusion_matrix(
                    true_j.ravel(), pred_j.ravel(),
                    labels=list(range(len(class_names)))  # Fix: ensure consistent label ordering
                )

                if SAVE_OPT:
                    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

                    axes[0].imshow(inputs[j].permute(1,2,0).cpu().numpy())
                    axes[0].set_title('Imagen Original', fontsize=12)
                    axes[0].axis('off')

                    axes[1].imshow(true_j, cmap=custom_cmap, norm=custom_norm, interpolation='nearest')
                    axes[1].set_title('Ground Truth (GT)', fontsize=12)
                    axes[1].axis('off')

                    axes[2].imshow(pred_j, cmap=custom_cmap, norm=custom_norm, interpolation='nearest')
                    axes[2].set_title('Predicción', fontsize=12)
                    axes[2].axis('off')

                    # Restaurada tu ruta y formato de guardado original
                    save_path = os.path.join(result_dir, 'predictions_multiclass', img_name[:-4] + '.png')
                    plt.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0.1)

                    plt.close(fig)

                coord_writer.writerow([img_name, str(ji)])

        mean_jaccard = np.mean(jaccard, axis=0)
        coord_writer.writerow(['MEAN', str(mean_jaccard)])

    print(f'Jaccard index per class: {mean_jaccard}')

    # --- VISUALIZACIÓN DE LA MATRIZ DE CONFUSIÓN (PORCENTUAL) ---
    fig, ax = plt.subplots(figsize=(12, 10))

    # 1. Cálculo de proporciones
    row_sums = cm_total.sum(axis=1)[:, np.newaxis]
    cm_normalized = np.divide(
        cm_total.astype(float),
        row_sums,
        out=np.zeros_like(cm_total, dtype=float),
        where=row_sums != 0
    )

    # 2. Convertir a porcentaje y REDONDEAR explícitamente a 2 decimales
    cm_percentage = np.round(cm_normalized * 100.0, 2)

    # 3. Crear el display
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_percentage, display_labels=class_names)

    # 4. Graficar forzando formato y reduciendo tamaño de fuente de los números (text_kw)
    disp.plot(
        cmap='Blues',
        values_format='.2f',
        ax=ax,
        xticks_rotation=45,
        text_kw={"fontsize": 8}  # <--- Esto evita que los números grandes solapen la celda
    )

    plt.title('Matriz de Confusión Acumulada Normalizada por Fila (%)')
    plt.tight_layout()

    cm_save_path = os.path.join(result_dir, 'confusion_matrix.png')
    plt.savefig(cm_save_path, dpi=300)
    plt.show()
    #plt.close(fig)
    # ------------------------------------------------------------

    return cm_total

### Entrenamiento


La función recibe como parámetros:

- __model__: la CNN que evaluar.
- __dataloader__: el cargador de los datos de test.
- __device__: el dispositivo que utilizar para el entrenamiento (GPU o CPU).
- __optimizer__: el optimizador.
- __lr_scheduler__: la política de modificación de la tasa de aprendizaje.
- __metrics__: las métricas para medir el rendimiento del sistema de segmentación.
- __bpath__: el directorio donde almacenar la mejor de las redes de segmentación.
- __num_classes__: el número de clases para la segmentación.
- __num_epochs__: el número de *epochs* durante los que entrenar.

__**IMPORTANTE:__ normalmente, cuando se realiza un procedimiento de _fine-tuning_ sobre una red ya pre-entrenada, la base de datos de que se dispone es pequeña y no se dispone de gran capacidad de computación (que permitiría usar *batches* más grandes). En estos casos es una buena práctica fijar los módulos de normalización de *batch* de la red (poniéndolos en modo `eval`).

In [ ]:
# CELDA 35
from sklearn.metrics import roc_auc_score, jaccard_score
from sklearn.preprocessing import LabelBinarizer


def set_bn_eval(mm):
    """Set BatchNorm modules to eval mode to freeze running stats during training."""
    if isinstance(mm, torch.nn.modules.batchnorm._BatchNorm):
        mm.eval()


def train_model(model, criterion, dataloaders, device, optimizer, lr_scheduler, metrics, bpath, model_name, num_classes=5, num_epochs=3):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_jaccard = 0

    fieldnames = ['epoch', 'Train_loss', 'Val_loss'] + \
        [f'Train_{m}' for m in metrics.keys()] + \
        [f'Val_{m}' for m in metrics.keys()]

    with open(os.path.join(bpath, 'log.csv'), 'w', newline='') as csvfile:
        csv.DictWriter(csvfile, fieldnames=fieldnames).writeheader()

    lb = LabelBinarizer()
    lb.fit(range(num_classes))

    for epoch in range(1, num_epochs + 1):
        checkpoint_path = os.path.join(bpath, f'{model_name}-epoch{epoch}.pth')

        if os.path.exists(checkpoint_path):
            print(f"> Loading checkpoint for epoch {epoch}")
            checkpoint = torch.load(checkpoint_path, weights_only=False)
            model.load_state_dict(checkpoint['state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            lr_scheduler.load_state_dict(checkpoint['scheduler'])
            best_jaccard = checkpoint['best_jaccard']
            print(f"> Checkpoint loaded for epoch {epoch}")
            continue

        print(f'Epoch {epoch}/{num_epochs}')
        print('-' * 10)

        batchsummary = {a: [0] for a in fieldnames}

        for phase in ['Train', 'Val']:
            if phase == 'Train':
                model.train()
            else:
                model.eval()
            model.apply(set_bn_eval)

            for sample in tqdm(iter(dataloaders[phase])):
                inputs = sample['image'].to(device)
                masks = sample['mask'].to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'Train'):
                    outputs = model(inputs)
                    loss = criterion(outputs['out'], masks)

                    y_pred = outputs['out']
                    y_prob = torch.nn.functional.softmax(y_pred, dim=1)
                    _, y_pred = torch.max(y_prob, dim=1)
                    y_pred = y_pred.data.cpu().numpy()

                    # Reshape prob tensor from (B, C, H, W) -> (C, B*H*W)
                    y_prob = np.reshape(
                        np.transpose(y_prob.data.cpu().numpy(), (1, 0, 2, 3)),
                        (num_classes, -1)
                    )
                    y_true = masks.data.cpu().numpy()

                    if num_classes == 2:
                        y_true = y_true > 0

                    for name, metric in metrics.items():
                        if name == 'jaccard_score':
                            if num_classes == 2:
                                ji = metric(y_true[:, 1, :, :].ravel(), y_pred.ravel(), average=None)
                            else:
                                ji = metric(y_true.ravel(), y_pred.ravel(), labels=np.unique(y_true), average=None)
                            # Exclude background class (index 0) from mean
                            batchsummary[f'{phase}_{name}'].append(np.mean(ji[1:]))

                        else:  # roc_auc_score
                            if num_classes == 2:
                                batchsummary[f'{phase}_{name}'].append(
                                    metric(y_true[:, 1, :, :].ravel(), y_prob[1:, :].T,
                                           average='micro', multi_class='ovr')
                                )
                            else:
                                batchsummary[f'{phase}_{name}'].append(
                                    metric(lb.transform(y_true.ravel()), y_prob.T,
                                           average='micro', multi_class='ovr')
                                )

                    if phase == 'Train':
                        loss.backward()
                        optimizer.step()

            batchsummary['epoch'] = epoch
            batchsummary[f'{phase}_loss'] = loss.item()
            print(f'{phase} Loss: {loss:.4f}')

        for field in fieldnames[3:]:
            batchsummary[field] = np.nanmean(batchsummary[field])
        print(batchsummary)

        with open(os.path.join(bpath, 'log.csv'), 'a', newline='') as csvfile:
            csv.DictWriter(csvfile, fieldnames=fieldnames).writerow(batchsummary)

        if batchsummary['Val_jaccard_score'] >= best_jaccard:
            best_jaccard = batchsummary['Val_jaccard_score']
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save({'state_dict': best_model_wts}, os.path.join(bpath, f'{model_name}_best.pth.tar'))

        lr_scheduler.step()

        torch.save({
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': lr_scheduler.state_dict(),
            'best_jaccard': best_jaccard,
        }, checkpoint_path)

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Highest Jaccard: {best_jaccard:.4f}')

    model.load_state_dict(best_model_wts)
    return model

En el caso de la segmentación multi-clase, la salida de la red es una imagen con tantos canales como clases consideradas para la segmentación. El criterio para el entrenamiento de la red es la función `CrossEntropyLoss` para clasificación multi-clase, ahora a nivel de píxel, para cada una de las imágenes a segmentar.

El entrenamiento de la red se realiza durante 8 epochs, reduciendo la tasa de aprendizaje a medida que se avanza en el entrenamiento. El código produce un archivo denominado _log.csv_ donde se puede analizar la variación de las funciones de pérdida en cada _epoch_ de entrenamiento, así como la correspondiente precisión y recall en el conjunto de test tanto para detección como para clasificación. Compruebe que las funciones de pérdida son algo ruidosas, y que las medidas de evaluación (área bajo la curva ROC e índice Jaccard) en validación van creciendo a medida que avanza el entrenamiento.

__**IMPORTANTE:__ note cómo el área ROC no es descriptiva de cómo avanza el proceso de entrenamiento para nuestro caso (desde el principio es muy elevada, por encima del 95%) y no está correlada con el índice Jaccard. Esto ocurre por el __desbalanceo__ de la base de datos (la cúpula óptica representa una parte muy pequeña de las imágenes). En bases de datos muy desbalanceadas la curva ROC no es un buen indicativo de la eficiencia de la segmentación ya que la importancia que da a las clases depende de su probabilidad de aparición.

In [ ]:
# custom weight initialization
def weights_init(m):
    if isinstance(m, torch.nn.Conv2d):
        torch.nn.init.xavier_normal_(m.weight,1.0)

model = get_deeplabv3(num_classes)
# Use gpu if available
model.to(device)

model_name='deeplabv3'

# Training stage
model.train()

# Create the experiment directory if not present
if not os.path.isdir(result_dir):
    os.mkdir(result_dir)

# Specify the loss function
criterion = torch.nn.CrossEntropyLoss(reduction='mean')# 3 classes

# Specify the optimizer with a lower learning rate for backbone
params_classifier = [p for p in model.classifier.parameters() if p.requires_grad]

params_backbone = [p for p in model.backbone.parameters() if p.requires_grad]

# Initialize the classifier-conv_layer weights, to adapt to the new paradigm (retinography vs natural images)
torch.manual_seed(manualSeed)
model.classifier.apply(weights_init)

# We apply a different lr to backbone and classifier parts
optimizer = torch.optim.Adam([
{'params': params_backbone},
{'params': params_classifier, 'lr': 1e-4}
], lr=1e-5)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                          step_size=step_size,
                                          gamma=0.1)

# Specify the evaluation metrics
metrics = {'jaccard_score': jaccard_score, 'auroc': roc_auc_score}

trained_model = train_model(model, criterion, dataloaders, device,
                        optimizer, lr_scheduler, bpath=result_dir, model_name=model_name,
                        metrics=metrics, num_classes=num_classes, num_epochs=num_epochs)

### Evaluación

Tras entrenar la red, se van a evaluar los resultados para el conjunto de test. En primer lugar, se cargan los pesos de la red entrenada en el modelo y se llama a la función de evaluación. Preste atención a los parámetros que recibe la función.

In [ ]:
# Inference
weights=torch.load(os.path.join(result_dir,model_name+'_best.pth.tar'))['state_dict']
model = get_deeplabv3(num_classes)
model.to(device)
model.load_state_dict(weights)
model.eval()
cm_total = test_segmentation_model(model, dataloaders, num_classes, class_names, result_dir, True, batchsize_test)

## Parte 3. Experimentos

### 1. Utilidad de las capas *atrous* y el ASPP

En este experimento, para poder observar la capacidad de representación del ASPP con capas *atrous*, se construye una red similar a Deeplab-V2 sobre la arquitectura que tenemos. En Deeplab-V2 las capas *atrous* se encuentran justamente por delante de la función de *loss*, de manera que sus activaciones están __directamente relacionadas__ con la segmentación de salida de la red. En nuestra red sin embargo, las capas *atrous* están __más alejadas__ de la función de *loss* de manera que es __más difícil analizar sus activaciones__. Para convertir nuestra arquitectura en la de Deeplab-V2, se realiza lo siguiente:

- Se elimina el bloque ASPP de Deeplab-V3, que contiene el *avg pooling*.
- Se añade el bloque ASPP de Deeplab-V2, con 4 convoluciones *atrous* de tasa $r=6, 12, 18$ y $24$. A las salidas de cada rama del ASPP se coloca una capa de convolución que convierte cada mapa de características al tamaño del mapa de salida (según el número de clases, en nuestro caso 2, *background* + cúpula óptica).
- Se suman las contribuciones de cada rama para generar la salida y se aplica la función de *loss*.

La siguiente figura muestra los cambios que se realizan para visualizar las activaciones.

<img src="https://www.tsc.uc3m.es/~matorres/images/apai/segmentation/deeplabv3v2.png" width="600pix">

De este modo, se pueden visualizar las salidas marginales de cada rama y ver cómo funcionan las convoluciones *atrous* a partir de sus activaciones.

- Analice las activaciones de cada rama para las imágenes de test. ¿Qué diferencias encuentra y a qué se deben? ¿Cómo contribuyen las convoluciones *atrous* y el ASPP a mejorar la segmentación? ¿Serán útiles si hay muchos objetos en la imagen (segmentación densa)?

In [ ]:
class Sum(torch.nn.Module):
    def __init__(self, num_classes):
        super(Sum, self).__init__()
        self.num_classes = num_classes
    def forward(self, x):
        x=x.unsqueeze(1)
        x=x.view(x.size(0),4,self.num_classes,x.size(3),x.size(4))
        x=torch.sum(x,dim=1)
        return x

def get_deeplabv2(num_classes=1):
    model=get_deeplabv3(num_classes=2)
    # Remove the last avg pooling
    model.classifier[0].convs=model.classifier[0].convs[:-1]
    # Change the convolutions from the remaining blocks r=6,12,28 and 24 and adding conv-layers to each branch
    modules = []
    modules.append(torch.nn.Conv2d(2048,256, kernel_size=3,stride=1,padding=6,dilation=6))
    modules.append(model.classifier[0].convs[0][1])
    modules.append(model.classifier[0].convs[0][2])
    modules.append(torch.nn.Conv2d(256, num_classes, kernel_size=1))
    model.classifier[0].convs[0]=torch.nn.Sequential(*modules)
    modules = []
    modules.append(torch.nn.Conv2d(2048,256, kernel_size=3,stride=1,padding=12,dilation=12))
    modules.append(model.classifier[0].convs[1][1])
    modules.append(model.classifier[0].convs[1][2])
    modules.append(torch.nn.Conv2d(256, num_classes, kernel_size=1))
    model.classifier[0].convs[1]=torch.nn.Sequential(*modules)
    modules = []
    modules.append(torch.nn.Conv2d(2048,256, kernel_size=3,stride=1,padding=18,dilation=18))
    modules.append(model.classifier[0].convs[2][1])
    modules.append(model.classifier[0].convs[2][2])
    modules.append(torch.nn.Conv2d(256, num_classes, kernel_size=1))
    model.classifier[0].convs[2]=torch.nn.Sequential(*modules)
    modules = []
    modules.append(torch.nn.Conv2d(2048,256, kernel_size=3,stride=1,padding=24,dilation=24))
    modules.append(model.classifier[0].convs[3][1])
    modules.append(model.classifier[0].convs[3][2])
    modules.append(torch.nn.Conv2d(256, num_classes, kernel_size=1))
    model.classifier[0].convs[3]=torch.nn.Sequential(*modules)
    # Sum the marginal predictions for each branch
    model.classifier[0].project=Sum(num_classes)
    model.classifier=model.classifier[0]
    return model

model_name='deeplabv2'

model = get_deeplabv2(num_classes)
print(model)
model.to(device)

# Training stage
model.train()

# Specify the loss function
criterion = torch.nn.CrossEntropyLoss(reduction='mean')# 3 classes

# Specify the optimizer with a lower learning rate for backbone
params_classifier = [p for p in model.classifier.parameters() if p.requires_grad]

params_backbone = [p for p in model.backbone.parameters() if p.requires_grad]

# Initialize the classifier-conv_layer weights, to adapt to the new paradigm (retinography vs natural images)
torch.manual_seed(manualSeed)
model.classifier.apply(weights_init)

# We apply a different lr to backbone and classifier parts
optimizer = torch.optim.Adam([
{'params': params_backbone},
{'params': params_classifier, 'lr': 1e-3}
], lr=1e-4)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                          step_size=step_size,
                                          gamma=0.1)

# Specify the evaluation metrics
metrics = {'jaccard_score': jaccard_score, 'auroc': roc_auc_score}

trained_model = train_model(model, criterion, dataloaders, device,
                        optimizer, lr_scheduler, bpath=result_dir, model_name=model_name,
                        metrics=metrics, num_classes=num_classes, num_epochs=num_epochs)

__**IMPORTANTE:__ la función externa `get_activations` proporciona una representación visual de las salidas del ASPP, que almacena en la carpeta `activations` en el directorio de resultados. Sus entradas son:

- __model__: la CNN que evaluar.
- __dataloaders__: el cargador de los datos de test.
- __device__: el dispositivo que utilizar para el entrenamiento (GPU o CPU).
- __result_dir__: el directorio de resultados.
- __batchsize_test__: el tamaño de batch para test, que debe ser 1.

__**IMPORTANTE:__ la función `get_activations` hace uso de los *hook* de Pytorch. Los *hook* de Pytorch son una serie de funciones que permiten modificar los datos de entrada o salida de alguna capa de la red durante la ejecución de la red. Esto permite tener acceso a datos intermedios de la red en tiempo de ejecución, así como realizar un *debug* controlado de la red neuronal. En concreto se usa un *register_forward_hook()*, se ejecuta tras el método *forward* de cualquier capa de la red y tiene acceso a sus entradas y salidas.

In [ ]:

import os
import sys
# Asegúrate de que estamos en el directorio correcto
print(f"Directorio actual: {os.getcwd()}")
print(f"¿external.py existe? {os.path.exists('external.py')}")

# Si no existe, agrégalo al path
if not os.path.exists('external.py'):
    print("\n✗ external.py no encontrado. Buscando...")
    for root, dirs, files in os.walk('.'):
        if 'external.py' in files:
            path = os.path.dirname(os.path.join(root))
            print(f"✓ Encontrado en: {path}")
            if path not in sys.path:
                sys.path.insert(0, path)
            break
else:
    print("✓ external.py encontrado en directorio actual")

In [ ]:
from external import get_activations
# Code to obtain the activations
weights=torch.load(os.path.join(result_dir,model_name+'_best.pth.tar'))['state_dict']
model = get_deeplabv2(num_classes)
model.to(device)
model.load_state_dict(weights)
model.eval()
get_activations(model, dataloaders, device, result_dir, batchsize_test)

### 2. Funciones de pérdida

En este apartado se van a describir situaciones  con la función de pérdida de la red para posteriores experimentos. A la hora de entrenar una red de segmentación, hay que tener en cuenta dos cosas principalmente: el desbalanceo de las clases; si existe, y el uso de estrategias de regularización de la función de pérdida pixelar. Ambas se definen a continuación.

#### Desbalanceo de las clases

El desbalanceo de las clases ocurre cuando algunas de las clases en la salida dominan sobre el resto (es decir, la proporción de píxeles pertenecientes a las distintas clases es muy desigual). Esto puede provocar que la red tienda a dar demasiada importancia a las clases más representadas a costa de reducir la importancia (o incluso hacer desaparecer) clases poco representadas. Esto se puede comprender mejor si se analiza a nivel de píxel: __la función de pérdida estándar aplica la misma importancia a todos los píxeles de la salida, independientemente de su clase, es decir, la red se va a centrar en clasificar cada uno de ellos correctamente, sea cual sea su clase. Sin embargo, si existe desbalanceo para las clases, un error en una clase poco representada (1 error sobre 10 píxeles, por ejemplo) será más perjudicial para el rendimiento del sistema que un error en una clase muy representada (1/1000)__. Si se une esto a que la red puede aprender mejor la clase muy representada porque tiene un mayor número de ejemplos de la misma, el resultado de la segmentación puede ser poco preciso

Como estrategia para mitigar esto se propone una muy sencilla: se basa en aplicar una serie de pesos a las clases en la función de pérdida según su probabilidad de aparición en el conjunto de entrenamiento. Esto provoca que la red no ponga el mismo énfasis en clasificar cada píxel, sino que un error en un píxel de una clase poco representada dará un valor de la función de pérdida mayor que en una clase muy representada, de manera que la red pone más énfasis en resolver los errores sobre la clase más "difícil". Sin embargo, esta estrategia es sensible a los pesos que se apliquen a las clases.

#### Estrategias de regularización

Por otra parte, la función de pérdida pixelar no tiene en cuenta ninguna dependencia entre los píxeles de la imagen (es decir, a la función contribuyen de igual manera todos los píxeles de la imagen; y entre ellos no se impone ninguna relación). Sin embargo, en el caso de la segmentación de imágenes existe una fuerte dependencia entre los píxeles:

- A nivel local, píxeles adyacentes a uno clasificado como perteneciente a la clase 'X' por ejemplo, tienen más probabilidad de ser de la clase 'X' que de la clase 'Y' en imágenes naturales (los objetos son continuos hasta que se llega a sus bordes). Existen estrategias sobre la función de pérdida que pueden reforzar la coherencia local en las segmentaciones (véase *'pairwise penalties'* en [6]).
- A nivel de imagen, existirán imágenes que la red segmente correctamente y otras en las que el desempeño sea menor. Existen funciones de pérdida que modelan esto (hacen que la red se centre en las imágenes más complejas) y combinadas con la función pixelar pueden resultar de ayuda, como la Dice Loss, en [8].
- En objetos no uniformes (con regiones con aspectos muy diferentes), pueden existir ciertos tipos de regiones que la red segmente muy bien y otras que sean complejas. Una función de pérdida que dé mayor importancia a los píxeles mal clasificados que a los correctamente clasificados puede ser útil. De este modo la red se centrará en clasificar correctamente dichos píxeles (lo que frecuentemente no afecta a los píxeles ya correctamente clasificados, que son más sencillos) y los resultados mejorarán. Esto equivale a hacer un procedimiento de *Hard Negative Mining* sobre los píxeles (es decir, centrarse en aquellos que resultan más difíciles para la red). La función Focal Loss [7] puede ser un buen ejemplo.

## [!] Parte 4. Trabajo autónomo del estudiante para la realización del proyecto

## Segmentación semántica de edificios afectados por desastres naturales

De este proyecto (si se elige) surge la segunda evaluación para la asignatura. Una vez comprendidos los fundamentos de la segmentación semántica con DeepLab-V3, el proyecto consistirá en realizar experimentos dirigidos a:

- Aplicar la segmentación semántica multi-clase al problema de evaluación del impacto de desastres naturales del primer proyecto, prediciendo para cada píxel de la imagen si pertenece al fondo, o a un edificio con alguno de los cuatro niveles de daño.
- Analizar los resultados y proporcionar estrategias de mejora sobre la arquitectura, el proceso de entrenamiento o la función de pérdida.

#### Detalles de la base de datos xBD

xBD es uno de los conjuntos de datos de evaluación de daños en edificios más grandes hasta la fecha. Contiene imágenes de alta resolución (menos de 0.8 m de GSD) que cubren 19 desastres naturales diferentes en todo el mundo (terremotos, inundaciones, incendios forestales, huracanes y tsunamis).

Para la segmentación semántica, cada imagen de 1024×1024 píxeles se acompaña de una **máscara de segmentación** a nivel de píxel con las siguientes clases:

| Valor en máscara | Significado |
|---|---|
| 0 | Fondo (*background*) |
| 1 | Edificio sin daño (*no-damage*) |
| 2 | Daño menor (*minor-damage*) |
| 3 | Daño mayor (*major-damage*) |
| 4 | Destruido (*destroyed*) |

### 4.1 Configuración

Cargue las librerías necesarias. Nótese que la mayor parte del código de entrenamiento y evaluación de las partes anteriores se puede reutilizar.

In [11]:
# ────────────────────────────────────────────────────────────────────────────
# CELDA 52 - SETUP KAGGLE: Imports y Configuración Inicial
# ────────────────────────────────────────────────────────────────────────────
import os
import glob
import json
import tifffile
from tqdm import tqdm
import numpy as np
import copy
import time
import torch
import torch.nn.functional as FT
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from torchvision.models.segmentation import deeplabv3_resnet101
from torchvision import transforms, utils
import torchvision.transforms.functional as F
from PIL import Image, ImageFile, ImageDraw
import cv2
import csv
import random
import matplotlib.pyplot as plt
from shapely.wkt import loads
from shapely.geometry import Polygon
from collections import defaultdict
from natsort import natsorted

# ─── Módulos del proyecto (Kaggle ya los encuentra gracias al sys.path) ─────
from xbd_dataset import xBDDataset, IMAGENET_STATS, DAMAGE_CLASSES, IDX_TO_CLASS
from augment import JointTransform, ABLATION_CONFIGS, visualize_augmentation
from sklearn.model_selection import train_test_split
from losses import get_loss
from arch import get_deeplabv3_xbd
from train_utils import train_model_xbd, test_segmentation_model_xbd, set_bn_eval

# Set random seed for reproducibility
manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)

# ¡AVISO TECH LEAD! 
# Activamos CuDNN y el benchmark para que la GPU en Kaggle vuele.
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

Random Seed:  999


### 4.2 Dataset xBD adaptado para segmentación de imágenes

A diferencia del Proyecto 1, donde el dataset devolvía un *parche centrado en un edificio* junto con su etiqueta de daño (tarea de clasificación), aquí necesitamos adaptar la clase `xBDDataset` para la tarea de **segmentación de imágenes**. Para ello, el dataset debe devolver, para cada imagen (o ventana de imagen), una máscara de segmentación semántica donde los píxeles de cada edificio aparezcan anotados con una etiqueta de daño.

La máscara devuelta es un `LongTensor` de forma `[H, W]` con valores enteros de 0 a 4.

Usaremos la clase `xBDDataset` mejorada del Proyecto 1 (modo `task="segmentation"`) como punto de partida.

In [12]:
# ────────────────────────────────────────────────────────────────────────────
# CELDA 54 — xBDDataset (Comprobación de variables)
# ────────────────────────────────────────────────────────────────────────────
print(f"xBDDataset disponible. Clases de daño: {DAMAGE_CLASSES}")
print("Estadísticas ImageNet:")
print(f"  mean = {IMAGENET_STATS['mean']}")
print(f"  std  = {IMAGENET_STATS['std']}")

xBDDataset disponible. Clases de daño: {'no-damage': 1, 'minor-damage': 2, 'major-damage': 3, 'destroyed': 4}
Estadísticas ImageNet:
  mean = [0.485 0.456 0.406]
  std  = [0.229 0.224 0.225]


In [ ]:
"""# ══════════════════════════════════════════════════════════════════════════════
# M1 — JointTransform (definido en augment.py, ya importado arriba)
# ══════════════════════════════════════════════════════════════════════════════
# Configuraciones de ablación predefinidas (ver augment.py:ABLATION_CONFIGS):
print("Configuraciones de ablación M1 disponibles:")
for name, cfg in ABLATION_CONFIGS.items():
    desc = cfg["description"]
    print(f"  · {name:<10s} → {desc}")

# Ejemplo de uso directo:
#   jt = JointTransform(hflip_p=0.5, vflip_p=0.5, rotation_degrees=15)
#   sample = jt(sample)
print("\n✓ JointTransform y ABLATION_CONFIGS disponibles desde augment.py")"""

Configuraciones de ablación M1 disponibles:
  · aug_none   → Sin augmentation (baseline)
  · aug_flips  → Flips H + V
  · aug_geo    → Flips + Rotación ±15°
  · aug_full   → Full Aug (Flips + Rot + Color + Blur)

✓ JointTransform y ABLATION_CONFIGS disponibles desde augment.py


### 4.3 Configuración del experimento y primeros modelos

Defina los hiperparámetros necesarios y cargue los conjuntos de datos de entrenamiento, validación y test. A continuación, implemente los primeros modelos para la realización del proyecto.

Para abordar la tarea de manera progresiva, se recomienda que comencéis implementando soluciones más sencillas y computacionalmente ligeras. Esto os permitirá validar que todo vuestro *pipeline* (carga de datos, entrenamiento y evaluación) funciona correctamente antes de añadir complejidad. Os sugerimos explorar los siguientes puntos:

* **Extracción de parches de menor tamaño:** Comenzad dividiendo las imágenes de alta resolución en parches más pequeños (ej. 64x64 o 128x128), de forma similar a lo que hicisteis en el proyecto de clasificación. Esto reducirá drásticamente el consumo de memoria y acelerará las pruebas iniciales.
* **Simplificación a segmentación binaria:** Antes de intentar predecir todos los niveles de daño (problema multiclase), probad a resolver un problema binario detectando únicamente "edificio" frente a "fondo" (*background*).
* **Arquitecturas *custom* sencillas:** En lugar de emplear modelos del estado del arte desde el principio, podéis diseñar y entrenar una arquitectura *Encoder-Decoder* sencilla desde cero. Os ayudará a comprender mejor el proceso de reducción y recuperación de la resolución espacial.
* **Data augmentation básico:** Implementad un proceso de *data augmentation* simple (ej. rotaciones, reflexiones o traslaciones). **Importante:** Recordad que en segmentación, cualquier transformación espacial aplicada a la imagen de entrada debe aplicarse exactamente igual a su máscara o *ground truth*.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CELDA 57 - SETUP KAGGLE: CONFIGURACIÓN DEFINITIVA
# ────────────────────────────────────────────────────────────────────────────
# Ruta del dataset en Kaggle (¡La ruta mágica que encontramos antes!)
data_dir        = "/kaggle/input/datasets/mariamontanet/xbd-uc3m/xBD_UC3M"

num_workers     = 2          # 2 está genial para Kaggle
img_size        = 128        # ¡256 para el RUN FINAL!
batchsize_train = 4          # Subimos a 8 para exprimir la GPU y acelerar
batchsize_test  = 1
num_classes     = 5

class_names = [
    "background",    # 0  ← fondo (sin edificio)
    "no-damage",     # 1
    "minor-damage",  # 2
    "major-damage",  # 3
    "destroyed",     # 4
]

num_epochs  = 5              # 6 épocas para encontrar nuestro mIoU de oro
step_size   = 5              # FIX: el LR se mantiene estable
result_dir  = "/kaggle/working/results_baseline" # Guardamos en la máquina virtual

# ¡SMOKE TEST APAGADO! Vamos a por todas las imágenes reales.
SMOKE_TEST = False
SMOKE_MAX_TRAIN = 30
SMOKE_MAX_VAL   = 10

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Dataset: {data_dir}")
print(f"Clases : {num_classes} → {class_names}")
print(f"Smoke test: {SMOKE_TEST} - ¡Entrenamiento Real Activado!")

Device : cuda:0
Dataset: /kaggle/input/datasets/mariamontanet/xbd-uc3m/xBD_UC3M
Clases : 5 → ['background', 'no-damage', 'minor-damage', 'major-damage', 'destroyed']
Smoke test: False - ¡Entrenamiento Real Activado!


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# BASELINE COMPARTIDO — Balanceo de clases con re-split estratificado
#
# Estrategia replicada del Proyecto 1 (Grupo 08):
#   1. Dataset de TEST cargado por separado (no se toca)
#   2. Splits train+val originales mezclados en un único pool
#   3. Re-split estratificado 85/15 por clase de daño dominante (sklearn)
#   4. WeightedRandomSampler: w = total / (num_clases × count)  [fórmula P1]
#
# Resultado: ambos splits tienen distribución de clases proporcional y el
# DataLoader de train sobremuestra clases minoritarias automáticamente.
# Cada miembro del equipo parte de esta base para sus experimentos (M1/M2/M3).
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter

# ── Clase auxiliar: aplica transform por split ────────────────────────────────
# Necesaria porque xBDDataset bake-in el transform en el constructor y no
# se puede cambiar por índice. Este wrapper lo aplica en __getitem__.
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base      = base_dataset
        self.indices   = np.array(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample = self.base[self.indices[idx]]
        if self.transform is not None:
            sample = self.transform(sample)
        return sample

# ── 1. Dataset de TEST (permanece separado, sin balanceo) ─────────────────────
max_test = SMOKE_MAX_VAL if SMOKE_TEST else 0
dataset_test = xBDDataset(
    data_dir   = data_dir,
    split      = ["test"],
    task       = "segmentation",
    patch_size = img_size,
    stats      = IMAGENET_STATS,
    transform  = None,
    max_size   = max_test,
)

# ── 2. Cargar train+val combinados (sin transform para poder re-dividir) ──────
print("\nCargando splits train+val combinados ...")
max_combined = (SMOKE_MAX_TRAIN + SMOKE_MAX_VAL) if SMOKE_TEST else 0
dataset_all = xBDDataset(
    data_dir   = data_dir,
    split      = ["train", "val"],
    task       = "segmentation",
    patch_size = img_size,
    stats      = IMAGENET_STATS,
    transform  = None,
    max_size   = max_combined,
)
print(f"  Total muestras combinadas: {len(dataset_all)}")

# ── 3. Clase dominante por muestra (clave de estratificación) ─────────────────
# = clase de daño más grave presente en el patch (sin cargar imágenes → rápido)
_IDX_NAME = {
    0: "background", 1: "no-damage", 2: "minor-damage",
    3: "major-damage", 4: "destroyed",
}
dominant_classes = np.array([
    max((b["label"] for b in s.get("buildings", []) if b["label"] > 0), default=0)
    for s in dataset_all.samples
])

print("\nDistribución ANTES del re-split:")
counts_all = Counter(dominant_classes.tolist())
total_all  = len(dominant_classes)
for cls_id in sorted(counts_all):
    n = counts_all[cls_id]
    print(f"  {_IDX_NAME.get(cls_id):15s}: {n:5d}  ({100*n/total_all:.1f}%)")

# ── 4. Re-split estratificado 85% train / 15% val ────────────────────────────
all_indices = np.arange(len(dataset_all))
train_idx, val_idx = train_test_split(
    all_indices,
    test_size    = 0.15,
    stratify     = dominant_classes,
    random_state = 42,
)

print(f"\nRe-split estratificado 85/15 (random_state=42):")
for split_name, idx_arr in [("Train", train_idx), ("Val", val_idx)]:
    c = Counter(dominant_classes[idx_arr].tolist())
    n = len(idx_arr)
    print(f"  {split_name}: {n:5d} muestras")
    for cls_id in sorted(c):
        pct = 100 * c[cls_id] / n
        print(f"    {_IDX_NAME.get(cls_id):15s}: {c[cls_id]:5d}  ({pct:.1f}%)")

# ── 5. Subsets con transform propio por split ─────────────────────────────────
# Baseline mínimo compartido: flips H+V (no contamina los experimentos de M1).
# M1 sustituye baseline_aug por sus configs de ablación en sus experimentos.
# M2 y M3 usan directamente este dataset_train / dataloaders["Train"].
baseline_aug = JointTransform(hflip_p=0.5, vflip_p=0.5)

dataset_train = TransformSubset(dataset_all, train_idx, transform=baseline_aug)
dataset_val   = TransformSubset(dataset_all, val_idx,   transform=None)

# ── 6. WeightedRandomSampler — misma fórmula que Proyecto 1 ──────────────────
# w_clase = total / (num_clases × count_clase)
train_dominant = dominant_classes[train_idx]
counts_train   = Counter(train_dominant.tolist())
total_train    = len(train_dominant)
num_cls        = 5   # 0=background, 1=no-damage, 2=minor, 3=major, 4=destroyed

cls_w = {
    cls: total_train / (num_cls * cnt)
    for cls, cnt in counts_train.items()
}
sample_weights = torch.DoubleTensor([cls_w[d] for d in train_dominant])

print("\nPesos de clase (WeightedRandomSampler):")
for cls_id in sorted(cls_w):
    print(f"  {_IDX_NAME.get(cls_id):15s}: {cls_w[cls_id]:.3f}×")

wrs_sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

# ── 7. DataLoaders finales ────────────────────────────────────────────────────
dataloaders = {
    "Train": DataLoader(
        dataset_train,
        batch_size  = batchsize_train,
        sampler     = wrs_sampler,    # ← WRS activo: balancea clases por batch
        num_workers = num_workers,
        pin_memory  = True,
    ),
    "Val": DataLoader(
        dataset_val,
        batch_size  = batchsize_test,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    ),
    "Test": DataLoader(
        dataset_test,
        batch_size  = batchsize_test,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
    ),
}

print(f"\n✅ Datasets y DataLoaders balanceados listos:")
print(f"  Train : {len(dataset_train):5d} muestras → {len(dataloaders['Train'])} batches  (WRS activo)")
print(f"  Val   : {len(dataset_val):5d} muestras → {len(dataloaders['Val'])} batches")
print(f"  Test  : {len(dataset_test):5d} muestras → {len(dataloaders['Test'])} batches")



[xBDDataset] split=['test']  task=segmentation  patch_size=256  samples=491
  no-damage       (idx=1):       0
  minor-damage    (idx=2):       0
  major-damage    (idx=3):       0
  destroyed       (idx=4):       0
  background      (idx=0):       0
  unlabelled     (idx=-1):   13260


Cargando splits train+val combinados ...

[xBDDataset] split=['train', 'val']  task=segmentation  patch_size=256  samples=2299
  no-damage       (idx=1):   34874
  minor-damage    (idx=2):    3370
  major-damage    (idx=3):    1996
  destroyed       (idx=4):    2198
  background      (idx=0):       0
  unlabelled     (idx=-1):       0

  Total muestras combinadas: 2299

Distribución ANTES del re-split:
  no-damage      :  1344  (58.5%)
  minor-damage   :   385  (16.7%)
  major-damage   :   238  (10.4%)
  destroyed      :   332  (14.4%)

Re-split estratificado 85/15 (random_state=42):
  Train:  1954 muestras
    no-damage      :  1143  (58.5%)
    minor-damage   :   327  (16.7%)
    major-damage   :   2

### 4.4 Adaptación del modelo Deeplab-V3 a xBD

Una vez tengáis un *pipeline* base funcionando y comprendáis las dificultades del problema, el objetivo es mejorar el rendimiento y acercaros a resultados del estado del arte, abordando los retos específicos de la base de datos xBD:

* **Aumento del contexto global (mayor resolución):** Escalad el tamaño de los parches de entrada (ej. 256x256 o 512x512). Para evaluar el estado de un edificio, la red necesita información del entorno y contexto que lo rodea, lo cual es difícil de capturar en parches muy pequeños.
* **Fine-tuning de modelos avanzados:** Dad el salto al uso de arquitecturas punteras para segmentación semántica, como **Deeplab-V3**, haciendo *fine-tuning* sobre un *backbone* profundo y pre-entrenado (por ejemplo, de la familia ResNet).
* **Módulos de extracción multiescala (ASPP):** Experimentad con el uso de convoluciones dilatadas (*atrous convolutions*) y módulos ASPP (*Atrous Spatial Pyramid Pooling*). Estas técnicas os permitirán capturar características a distintas escalas utilizando múltiples tasas de dilatación (ej. $r=6, 12, 18$) sin perder resolución espacial.

### 4.5 Pesos de clase y configuraciones de loss

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# M2 — Cálculo de pesos de clase a nivel de PÍXEL (no de patch)
# Se cachea en disco para no recomputar en cada sesión de Kaggle
# ══════════════════════════════════════════════════════════════════════════════
import os, torch
from losses import compute_class_weights

CACHE_PATH = f"/kaggle/working/repo/pixel_counts_train_{img_size}.pt"

if os.path.exists(CACHE_PATH):
    counts = torch.load(CACHE_PATH)
    print(f"Cargados counts cacheados: {counts.tolist()}")
else:
    print("Calculando pixel counts del train set (1-2 min)...")
    counts = torch.zeros(num_classes)
    # Iteramos sobre el dataset base (sin transform) — counts no dependen de aug
    base = dataset_train.base if hasattr(dataset_train, "base") else dataset_train
    for i in tqdm(range(len(dataset_train))):
        idx = dataset_train.indices[i] if hasattr(dataset_train, "indices") else i
        m = base[idx]["mask_patch"]
        for c in range(num_classes):
            counts[c] += (m == c).sum().item()
    torch.save(counts, CACHE_PATH)
    print(f"Counts guardados: {counts.tolist()}")

# Imprime distribución
total = counts.sum()
print("\nDistribución de píxeles en TRAIN:")
for i, n in enumerate(class_names):
    print(f"  {n:15s}: {int(counts[i]):>10d}  ({100*counts[i]/total:5.2f}%)")

# Pesos suaves (sqrt_inv_freq) — son los que usaremos en weighted_ce, focal, combo
class_weights = compute_class_weights(
    counts, scheme="sqrt_inv_freq", num_classes=num_classes, device=device,
)
print(f"\nPesos sqrt_inv_freq: {class_weights.cpu().tolist()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# M2 — Configuraciones de ablación de funciones de pérdida
# ══════════════════════════════════════════════════════════════════════════════
from losses import get_loss

LOSS_CONFIGS = {
    "ce": {
        "loss_factory": lambda: get_loss("ce"),
        "description": "CE plana (baseline)",
    },
    "weighted_ce": {
        "loss_factory": lambda: get_loss("weighted_ce", weight=class_weights),
        "description": "CE con pesos sqrt_inv_freq",
    },
    "dice": {
        "loss_factory": lambda: get_loss("dice", ignore_background=True),
        "description": "Dice (sin background)",
    },
    "focal": {
        "loss_factory": lambda: get_loss("focal", weight=class_weights, gamma=2.0),
        "description": "Focal γ=2, α=sqrt_inv_freq",
    },
    "focal_tversky": {
        "loss_factory": lambda: get_loss("focal_tversky",
                                         alpha=0.3, beta=0.7, gamma=0.75,
                                         ignore_background=True),
        "description": "Focal-Tversky α=0.3 β=0.7 γ=0.75",
    },
    "combo": {
        "loss_factory": lambda: get_loss("combo", weight=class_weights,
                                         alpha=0.5, dice_ignore_background=True),
        "description": "Combo 0.5·CE_w + 0.5·Dice",
    },
}

print(f"Configuraciones M2 definidas: {len(LOSS_CONFIGS)}")
for k, v in LOSS_CONFIGS.items():
    print(f"  · {k:<15s} → {v['description']}")

### 4.6 Entrenamiento

La función de entrenamiento seguirá el mismo esquema que en las partes anteriores del Proyecto 2A.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# M2 — BUCLE DE ABLACIÓN
# Para cada loss: entrena 5 epochs, evalúa en val, guarda resultados
# ══════════════════════════════════════════════════════════════════════════════
import gc
from sklearn.metrics import jaccard_score

# ─── Config ablación (overrides del baseline) ─────────────────────────────────
ABL_IMG_SIZE   = 128       # protocolo del README
ABL_BATCH      = 4         # protocolo del README
ABL_EPOCHS     = 5
ABL_BASE_DIR   = "/kaggle/working/results_m2"
os.makedirs(ABL_BASE_DIR, exist_ok=True)

# IMPORTANTE: si tu img_size del baseline ya es 128, puedes saltar la
# reconstrucción de dataloaders. Si era 256, vuelve a construir
# dataloaders_abl con patch_size=128 para esta ablación.
# Yo asumo que reusas tus dataloaders actuales:
dataloaders_abl = dataloaders   # cambia a la versión 128 si la creaste aparte

# ─── Resultados ───────────────────────────────────────────────────────────────
m2_results = {}

for cfg_name, cfg in LOSS_CONFIGS.items():
    print(f"\n{'='*70}")
    print(f"  M2 Config: {cfg_name}  —  {cfg['description']}")
    print(f"{'='*70}")

    cfg_dir = os.path.join(ABL_BASE_DIR, cfg_name)
    os.makedirs(cfg_dir, exist_ok=True)

    # ── Reproducibilidad: re-fijar semillas en CADA config ────────────────
    random.seed(manualSeed)
    np.random.seed(manualSeed)
    torch.manual_seed(manualSeed)
    torch.cuda.manual_seed_all(manualSeed)

    # ── Modelo fresco ─────────────────────────────────────────────────────
    model = get_deeplabv3_xbd(num_classes)
    model.to(device)

    # ── Loss según config ─────────────────────────────────────────────────
    criterion = cfg["loss_factory"]()

    # ── Optimizer + scheduler (idénticos para todas las configs) ──────────
    params_backbone   = [p for p in model.backbone.parameters()   if p.requires_grad]
    params_classifier = [p for p in model.classifier.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": params_backbone,   "lr": 5e-5},
        {"params": params_classifier, "lr": 5e-4},
    ], weight_decay=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=ABL_EPOCHS, eta_min=1e-6)

    metrics = {"jaccard_score": jaccard_score}

    # ── Entrenar ──────────────────────────────────────────────────────────
    trained = train_model_xbd(
        model, criterion, dataloaders_abl, device,
        optimizer, lr_scheduler, metrics=metrics,
        bpath=cfg_dir, model_name=f"deeplabv3_{cfg_name}",
        num_classes=num_classes, num_epochs=ABL_EPOCHS,
        use_aux=False,
    )

    # ── Evaluar en val (lo usamos como test) ──────────────────────────────
    dataloaders_abl["Test"] = dataloaders_abl["Val"]
    cm, iou_per_class = test_segmentation_model_xbd(
        trained, dataloaders_abl, device, num_classes, class_names,
        cfg_dir, SAVE_OPT=False,           # no guardes 345 PNGs por config
        batch_size=batchsize_test,
        stats=IMAGENET_STATS,
    )

    m2_results[cfg_name] = {
        "description":   cfg["description"],
        "iou_per_class": iou_per_class,           # array (4,) — clases 1..4
        "miou_fg":       float(np.mean(iou_per_class)),
        "cm":            cm,
    }
    print(f"\n>>> {cfg_name}: mIoU foreground = {m2_results[cfg_name]['miou_fg']:.4f}")

    # ── Limpieza de memoria GPU ───────────────────────────────────────────
    del model, trained, optimizer, lr_scheduler, criterion
    gc.collect()
    torch.cuda.empty_cache()

print("\n\nAblación M2 completada.")

### 4.7 Comparativa de funciones de pérdida

Evalúe el modelo de forma similar a como se ha realizado en la primera parte del notebook:

- Obtenga los resultados para el conjunto de test.
- Evalúe la influencia del umbral en la inferencia.
- Evalúe la utilidad de las capas atrous y el ASPP.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# M2 — Comparativa final: tabla + gráfico de barras agrupadas por clase
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

cfg_names    = list(m2_results.keys())
descriptions = [m2_results[n]["description"] for n in cfg_names]
mious        = [m2_results[n]["miou_fg"]      for n in cfg_names]
ious_matrix  = np.array([m2_results[n]["iou_per_class"] for n in cfg_names])  # (n_cfg, 4)

fg_classes = class_names[1:]  # 'no-damage', 'minor', 'major', 'destroyed'

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── (1) mIoU por config ───────────────────────────────────────────────────────
colors = plt.cm.tab10(np.linspace(0, 1, len(cfg_names)))
bars = axes[0].bar(cfg_names, mious, color=colors, edgecolor="black")
axes[0].set_ylabel("mIoU foreground (4 clases)")
axes[0].set_title("M2 — mIoU por función de pérdida", fontweight="bold")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_ylim(0, max(mious)*1.25)
for b, v in zip(bars, mious):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f"{v:.3f}", ha="center", fontweight="bold")
# Marca el ganador
best_idx = int(np.argmax(mious))
bars[best_idx].set_edgecolor("red")
bars[best_idx].set_linewidth(3)

# ── (2) IoU por clase, barras agrupadas ───────────────────────────────────────
x = np.arange(len(fg_classes))
width = 0.8 / len(cfg_names)
for i, (cfg, ious) in enumerate(zip(cfg_names, ious_matrix)):
    axes[1].bar(x + i*width - 0.4 + width/2, ious, width,
                label=cfg, color=colors[i], edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(fg_classes, rotation=0)
axes[1].set_ylabel("IoU por clase")
axes[1].set_title("M2 — IoU por clase × config", fontweight="bold")
axes[1].legend(fontsize=8, ncol=2, loc="upper right")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ABL_BASE_DIR, "M2_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────────
print("\n" + "="*92)
print(f"  {'Config':<15} | {'no-damage':>10} | {'minor':>8} | {'major':>8} | "
      f"{'destroy':>8} | {'mIoU':>7}")
print("="*92)
baseline = mious[0]
for cfg, ious, miou in zip(cfg_names, ious_matrix, mious):
    delta = miou - baseline
    delta_s = f"({'+' if delta>=0 else ''}{delta:.3f})"
    marker = "  ← BEST" if cfg == cfg_names[best_idx] else ""
    print(f"  {cfg:<15} | {ious[0]:>10.4f} | {ious[1]:>8.4f} | {ious[2]:>8.4f} | "
          f"{ious[3]:>8.4f} | {miou:>7.4f} {delta_s}{marker}")
print("="*92)

### 4.8 Discusión y extensiones propuestas

A continuación se proponen extensiones adicionales al experimento base, siguiendo las sugerencias del enunciado:

#### a) *Data augmentation* para segmentación
Las técnicas de aumento de datos usadas en el primer proyecto también pueden ser útiles aquí. Se pueden considerar transformaciones, como rotaciones, recortes aleatorios o cambios de escala, siempre aplicando la misma transformación a la imagen y a la máscara.

#### b) Modificar la estructura del clasificador de la red (backbone)
Se puede modificar la estructura del clasificador de la red para adecuarlo a la tarea de segmentación con la que se trabaja (muestre especial atención al *receptive field* de la red). Además, se puede utilizar el bloque de clasificación auxiliar de la red (`aux_classifier`), modificando su estructura y añadiendo una función de pérdida para el caso de segmentación multi-clase.

#### c) Funciones de pérdida y estrategias de regularización
A partir de las ideas proporcionadas en el apartado de funciones de pérdida, puede implementar alguna de las estrategias de regularización que se han descrito u otras que considere oportunas. Justifique su utilidad para la tarea que se propone. Recuerde que la base de datos xBD está fuertemente desbalanceada (los píxeles de edificios destruidos son una minoría frente a los intactos o el fondo). Puede evaluar el uso de funciones de coste avanzadas diseñadas para mitigar este problema, como *Focal Loss*, *Dice Loss* o la *Cross-Entropy* ponderada.

#### d) Evaluación por tipo de desastre
El dataset xBD contiene imágenes correspondientes a diferentes desastres naturales (ver carpetas y ficheros en la base de datos). Se puede estratificar la evaluación por tipo de desastre para analizar qué escenarios son más difíciles para el modelo.

__**IMPORTANTE:__ si encuentra problemas para entrenar la red Deeplab-V3 con backbone ResNet-101 o el entrenamiento es demasiado lento y desea agilizar los experimentos, el siguiente fragmento de código implementa Deeplab-V3 con un backbone mucho más ligero, de ResNet-18. Puede utilizarla como red *baseline* para el apartado autónomo. Tenga en cuenta que al no ser una red pre-entrenada en una base de datos para segmentación los resultados serán algo peores, especialmente en las activaciones *atrous* de la red, que visualmente no serán tan claras como en la red tratada.

## 5. Criterios de evaluación

- Presentación (Fecha indicada en la entrega del proyecto en Aula Global). Este día cada grupo de estudiantes tendrá un turno de 5 minutos de preguntas (máximo 5 minutos de presentación) sobre el apartado de trabajo autónomo con ayuda de un máximo de 5 transparencias.
  
- Informe + código. El grupo de estudiantes entregará un breve informe (5 páginas máximo, incluyendo descripciones, tablas, figuras y referencias) donde describirán los aspectos más importantes de la solución propuesta. El objetivo es que el grupo de estudiantes describa los análisis y extensiones que ha planteado al modelo y justifique su objetivo y utilidad de manera breve. Asimismo, se proporcionará el código utilizado para los experimentos (bien sobre este mismo notebook, en formato `.ipynb` o bien en código Python, en formato `.py`).

La fecha límite de entrega del fichero de código y el informe es la fecha indicada en la entrega del proyecto en Aula Global.